# Preparing Text Data for LLMs

A Large Language Model (LLM), like a calculator, can't understand words directly; it only understands numbers. Before we can train a model like GPT, we must first translate our text-based training data into this numerical format. By the end of this notebook, you will understand how to convert raw text into a stream of integer "tokens" and save it in a highly efficient binary format, perfectly prepared for a training loop.

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import re

# Set a consistent style for plots
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

### The Core Idea: From Prose to a Grocery List

Imagine you're trying to teach a robot chef to cook. You can't just give it a recipe book full of descriptive prose like "sauté the onions until they are translucent and fragrant." The robot doesn't understand "translucent" or "fragrant." It needs a precise, unambiguous list of instructions and ingredients.

1.  **The Prose:** "Heat one tablespoon of olive oil."
2.  **The Robot's Grocery List:** `[ITEM_ID_HEAT, ITEM_ID_ONE_TBSP, ITEM_ID_OLIVE_OIL]`

This is exactly what we do for an LLM. The raw text is the prose. The process of converting it into a list of numerical IDs is called **tokenization**. Each unique word or piece of a word (a "token") gets a unique integer ID, like an item number in a grocery store.

The model doesn't see "hello world"; it sees `[15339, 2964, 0]`. Our goal in this notebook is to become the "prep cook" who takes the beautiful, messy human language and converts it into the clean, structured, numerical format that the LLM can consume. We'll then store this list of numbers in a compact binary file, which is like putting all our prepped ingredients onto a single, long conveyor belt, ready for the assembly line (the training loop).

In [ ]:
# A simplified, mock BPE tokenizer for demonstration purposes.
# In a real-world scenario, you would use a library like tiktoken or SentencePiece.
class SimpleBPE:
    def __init__(self, vocab, merges):
        self.vocab = {s: i for i, s in enumerate(vocab)}
        self.merges = {tuple(k.split()): v for k, v in merges.items()}
        # The reverse mapping from integer ID back to token string
        self.ivocab = {i: s for s, i in self.vocab.items()}

    def encode(self, text):
        # Start with a list of single characters
        tokens = list(text)
        
        while True:
            # Find the first pair of tokens that can be merged
            min_pos = -1
            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i+1])
                if pair in self.merges:
                    min_pos = i
                    break
            
            # If no more merges are possible, we're done
            if min_pos == -1:
                break
            
            # Merge the pair into a new token
            pair_to_merge = (tokens[min_pos], tokens[min_pos+1])
            new_token = self.merges[pair_to_merge]
            tokens = tokens[:min_pos] + [new_token] + tokens[min_pos+2:]

        # Convert token strings to integer IDs
        return [self.vocab[token] for token in tokens]

def prepare_data(text, tokenizer, output_filename):
    """Tokenizes text and saves it to a binary file."""
    # Encode the entire text document into a list of token IDs
    token_ids = tokenizer.encode(text)
    
    # Convert the list to a NumPy array with a specific data type
    # uint16 is a good choice for vocabularies up to 65,535
    token_array = np.array(token_ids, dtype=np.uint16)
    
    # Write the array to a binary file
    token_array.tofile(output_filename)
    print(f"Saved {len(token_array)} tokens to {output_filename}")

### Walking Through the Code

Let's break down our data preparation pipeline step-by-step.

**1. `SimpleBPE` Class**
This is our mock tokenizer. Real tokenizers like Byte-Pair Encoding (BPE) are trained on massive datasets to learn the most efficient way to merge characters into common sub-words. We're faking it with a pre-defined set of rules.

-   `__init__(self, vocab, merges)`: The tokenizer is initialized with a `vocab` (a list of all possible final tokens) and `merges` (a dictionary telling it which pairs to combine, e.g., `'h' 'e' -> 'he'`). We create two dictionaries: one mapping tokens to integer IDs (`vocab`) and one for the reverse (`ivocab`).
-   `encode(self, text)`: This is the core logic.
    -   It starts by breaking the text into its most basic parts: individual characters. `text="hello"` becomes `['h', 'e', 'l', 'l', 'o']`.
    -   It then enters a `while` loop, repeatedly scanning the list of tokens for a pair that exists in our `merges` rules.
    -   If it finds a mergeable pair (e.g., `'h'` and `'e'`), it replaces them with the new merged token (`'he'`). The list becomes `['he', 'l', 'l', 'o']`.
    -   The loop continues until no more merges can be made.
    -   Finally, it looks up each final token in the `self.vocab` dictionary to get its integer ID and returns the list of IDs.

**2. `prepare_data` Function**
This function orchestrates the whole process.

-   `token_ids = tokenizer.encode(text)`: It takes the raw text and uses our tokenizer to convert it into a list of integers.
-   `token_array = np.array(token_ids, dtype=np.uint16)`: We convert the Python list into a NumPy array. We specify `dtype=np.uint16` to control the exact memory usage. A `uint16` is an unsigned 16-bit integer, which can hold values from 0 to 65,535. This is large enough for typical vocabulary sizes (like GPT-2's ~50,000) and uses only 2 bytes per token, which is very memory-efficient.
-   `token_array.tofile(output_filename)`: This is the final, crucial step. `tofile()` writes the NumPy array to disk as a raw, compact binary file. There's no extra formatting, just a continuous stream of 16-bit integers. This format is incredibly fast to load during training.

In [ ]:
# --- Demonstration ---

# 1. Define a minimal vocabulary and merge rules for our tokenizer
# A real vocab would have ~50,000 tokens.
vocab = ['h', 'e', 'l', 'o', ' ', 'w', 'r', 'd', '!', 'he', 'hel', 'lo', 'wor', 'orld']
merges = {
    "h e": "he",
    "he l": "hel",
    "l o": "lo",
    "w o": "wo", # Note: this merge won't be used for "world"
    "o r": "or", # Note: this won't be used for "world"
    "wor l": "worl", # Note: this won't be used for "world"
    "w or": "wor",
    "o rld": "orld",
}

# 2. Create some sample text data
sample_text = "hello world!"
with open("input.txt", "w") as f:
    f.write(sample_text)

# 3. Instantiate our tokenizer and prepare the data
bpe_tokenizer = SimpleBPE(vocab, merges)
prepare_data(sample_text, bpe_tokenizer, "train.bin")

# 4. Verify the process
# Load the binary file back into a NumPy array
loaded_tokens = np.fromfile("train.bin", dtype=np.uint16)
print(f"\nLoaded tokens from file: {loaded_tokens}")

# Decode the tokens back to text to ensure it's correct
decoded_text = "".join([bpe_tokenizer.ivocab[i] for i in loaded_tokens])
print(f"Decoded text: '{decoded_text}'")

# Check if the decoded text matches the original
assert decoded_text == sample_text

# Clean up created files
os.remove("input.txt")
os.remove("train.bin")

### Going Deeper: Why a Single, Giant Binary File?

A non-obvious design decision in `nanoGPT` (and other training frameworks) is to concatenate all training documents into a single, massive binary file of token IDs. Why not keep them as separate files or in a structured database?

The answer is **training efficiency**.

During training, the model needs to be fed chunks of data of a fixed size, called the `context_window` (e.g., 1024 tokens). The fastest way to get a random chunk of 1024 tokens from a 100-billion-token dataset is to:
1.  Pick a random starting position in the giant file (e.g., byte number 8,456,321,000).
2.  Read the next `1024 * 2` bytes (since each token is a `uint16`, which takes 2 bytes).

This operation is incredibly fast because it involves a simple `seek` and `read` from the disk, with no overhead from parsing file structures, directories, or databases. The giant binary file acts like a massive, contiguous array stored on disk.

In the real `nanoGPT` code, this is often handled by `numpy.memmap`, which is a memory-mapping tool. It lets you treat a file on disk *as if* it were a NumPy array in RAM, without actually loading the entire (potentially terabyte-sized) file into memory. The operating system intelligently handles paging data from disk to RAM as you access different parts of the array. Our use of `np.fromfile` is a simpler version of this idea for datasets that can fit in memory.

In [ ]:
# --- Visualization of the Data Preparation Pipeline ---

# Let's visualize the transformation for the sentence "hello world!"
sentence = "hello world!"

# 1. Text
text_stage = sentence

# 2. Tokenization (showing intermediate merges)
tokens = list(sentence) # -> ['h', 'e', 'l', 'l', 'o', ' ', 'w', 'o', 'r', 'l', 'd', '!']
# merge h+e -> he
tokens_stage1 = ['he', 'l', 'l', 'o', ' ', 'w', 'o', 'r', 'l', 'd', '!']
# merge he+l -> hel
tokens_stage2 = ['hel', 'l', 'o', ' ', 'w', 'o', 'r', 'l', 'd', '!']
# merge l+o -> lo
tokens_stage3 = ['hel', 'lo', ' ', 'w', 'o', 'r', 'l', 'd', '!']
# merge w+or -> wor
tokens_stage4 = ['hel', 'lo', ' ', 'wor', 'l', 'd', '!']
# merge o+rld -> orld
tokens_stage5 = ['hel', 'lo', ' ', 'wor', 'ld', '!'] # Note: 'd' and '!' are not merged
final_tokens = ['hel', 'lo', ' ', 'wor', 'ld', '!'] # Final set of tokens after all merges.

# 3. Integer IDs
token_ids = bpe_tokenizer.encode(sentence)

# 4. Binary Representation (as uint16, 2 bytes per ID)
binary_representation = np.array(token_ids, dtype=np.uint16).tobytes()

# --- Plotting ---
fig, axs = plt.subplots(4, 1, figsize=(12, 10))
fig.suptitle("The Data Preparation Pipeline", fontsize=16)

# Stage 1: Raw Text
axs[0].text(0.5, 0.5, f"'{text_stage}'", ha='center', va='center', fontsize=20, fontfamily='monospace')
axs[0].set_title("1. Raw Text String", loc='left')
axs[0].axis('off')

# Stage 2: Tokenization
axs[1].text(0.5, 0.5, str(final_tokens), ha='center', va='center', fontsize=20, fontfamily='monospace', color='darkgreen')
axs[1].set_title("2. Tokenization (into strings)", loc='left')
axs[1].axis('off')

# Stage 3: Integer IDs
axs[2].text(0.5, 0.5, str(token_ids), ha='center', va='center', fontsize=20, fontfamily='monospace', color='navy')
axs[2].set_title("3. Conversion to Integer IDs (uint16)", loc='left')
axs[2].axis('off')

# Stage 4: Binary File Representation
axs[3].set_title("4. Stored as a Raw Binary File (Bytes)", loc='left')
axs[3].set_xlim(-1, len(binary_representation) + 1)
axs[3].set_ylim(0, 2)
for i, byte_val in enumerate(binary_representation):
    # Each ID is 2 bytes (little-endian)
    is_high_byte = (i % 2 != 0)
    color = 'skyblue' if not is_high_byte else 'lightblue'
    rect = Rectangle((i, 0.5), 1, 1, facecolor=color, edgecolor='black')
    axs[3].add_patch(rect)
    axs[3].text(i + 0.5, 1, f"{byte_val:02x}", ha='center', va='center', fontfamily='monospace', fontsize=12)
axs[3].axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

### Summary

In this notebook, we built a miniature version of the `nanoGPT` data preparation pipeline. We saw how a language model can't work with raw text and requires a numerical representation.

**What we built:**
- A simple BPE-like tokenizer to break text into sub-word units.
- A script to convert these text tokens into integer IDs.
- A process to save these IDs into a flat, efficient binary file.

**Key Takeaways:**
-   **Tokenization is Translation:** It's the essential step of translating human language into the mathematical language of neural networks.
-   **Data Type Matters:** Choosing a compact data type like `uint16` saves significant disk space and memory, which is critical for large datasets.
-   **File Format Impacts Performance:** A simple, contiguous binary file is the most efficient format for quickly grabbing random batches of data during training.

**What's Next?**
Now that we have our data prepped and ready on the "conveyor belt," we can finally start building the machine that will process it. In the next notebook, *Deconstructing the GPT Model Architecture*, we will start assembling the core components of the Transformer model that will learn from this numerical data.